
# Gaussian Belief Propagation

This notebook concludes the **Inference in Gaussian Models** section.

In the previous notebook, exact inference was performed using **Gaussian Variable Elimination**:

```text
Combine factors
      ↓
Eliminate hidden variables
      ↓
Obtain query distribution
```

Belief Propagation expresses the same inference process in a different way.

Instead of eliminating variables centrally, neighboring nodes exchange local messages.

The high-level idea is:

```text
Factor graph
    ↓
Local messages
    ↓
Information propagates
    ↓
Marginal beliefs
```

For tree-structured Gaussian graphical models, Gaussian Belief Propagation produces exact marginal beliefs.

This notebook develops:

- Gaussian factor graphs,
- variable-to-factor messages,
- factor-to-variable messages,
- canonical-form messages,
- belief computation,
- a worked chain example,
- a complete implementation,
- comparison with Gaussian Variable Elimination.



## 1. Big Picture

The relationship between the two exact-inference algorithms is:

```text
                Gaussian Inference
                       │
            ┌──────────┴──────────┐
            │                     │
            ▼                     ▼
 Variable Elimination      Belief Propagation
            │                     │
   eliminate variables       pass messages
            │                     │
            └──────────┬──────────┘
                       ▼
               Same marginal belief
```

On trees, the two approaches are mathematically equivalent.

The difference is computational organization.



## 2. Factor Graph Reminder

A factor graph contains two kinds of nodes:

```text
Variable nodes      Factor nodes
     X                   f
```

For example:

```text
X ─── fXY ─── Y ─── fYZ ─── Z
```

The factors encode local probabilistic relationships.

Belief Propagation sends messages along the edges of this graph.

There are two message types:

```text
Variable → Factor
Factor   → Variable
```



## 3. Why Canonical Form?

A Gaussian message can be represented as

\[
m(x)
\propto
\exp
\left(
-\frac12 x^\top \Lambda x
+
\eta^\top x
\right).
\]

This message is completely characterized by

\[
(\eta,\Lambda).
\]

Canonical form is ideal here because multiplying Gaussian messages becomes

\[
\eta
=
\eta_1+\eta_2,
\]

\[
\Lambda
=
\Lambda_1+\Lambda_2.
\]

So information arriving from multiple neighbors simply **adds**.


In [ ]:

from __future__ import annotations

from dataclasses import dataclass

import numpy as np

np.set_printoptions(
    precision=4,
    suppress=True,
)



# 4. Gaussian BP Messages

For this notebook, each variable is scalar.

A message therefore needs only:

- one information value \(\eta\),
- one precision value \(\Lambda\).

We represent the message explicitly rather than relying on a larger canonical-Gaussian abstraction.


In [ ]:

@dataclass
class GaussianBPMessage:
    information: float
    precision: float

    def __post_init__(self) -> None:
        self.information = float(
            self.information
        )

        self.precision = float(
            self.precision
        )

        if self.precision < 0.0:
            raise ValueError(
                "Message precision cannot be negative."
            )

    @property
    def is_uninformative(self) -> bool:
        return np.isclose(
            self.precision,
            0.0,
        )

    def to_moment(
        self,
    ) -> tuple[float, float]:
        if self.is_uninformative:
            raise ValueError(
                "An uninformative message has no finite "
                "mean or variance."
            )

        variance = 1.0 / self.precision

        mean = (
            variance
            * self.information
        )

        return mean, variance



An uninformative Gaussian message is represented by

\[
\eta=0,
\qquad
\Lambda=0.
\]

It contributes no information when messages are added.



# 5. Variable-to-Factor Messages

Suppose variable \(X\) is connected to several factors.

The message from \(X\) to factor \(f\) is the product of all incoming factor-to-variable messages **except the message from \(f\)**:

\[
m_{X\rightarrow f}(x)
\propto
\prod_{g\in N(X)\setminus f}
m_{g\rightarrow X}(x).
\]

Because canonical Gaussian multiplication is addition,

\[
\boxed{
\Lambda_{X\rightarrow f}
=
\sum_{g\in N(X)\setminus f}
\Lambda_{g\rightarrow X}
}
\]

and

\[
\boxed{
\eta_{X\rightarrow f}
=
\sum_{g\in N(X)\setminus f}
\eta_{g\rightarrow X}.
}
\]



## 5.1 Intuition

A variable tells a neighboring factor:

> "Here is everything I currently know from my other neighbors."

The destination factor's own message is excluded to avoid immediately sending the same information back.



## 5.2 Variable-to-Factor Flowchart

```text
Incoming message 1 ─┐
                    │
Incoming message 2 ─┼──► Variable X ───► Factor f
                    │
Incoming message 3 ─┘

Exclude message from f

Then add:
η values
Λ values
```


In [ ]:

def variable_to_factor_message(
    incoming_messages: list[GaussianBPMessage],
) -> GaussianBPMessage:
    information = sum(
        message.information
        for message in incoming_messages
    )

    precision = sum(
        message.precision
        for message in incoming_messages
    )

    return GaussianBPMessage(
        information=information,
        precision=precision,
    )



# 6. Factor-to-Variable Messages

This is the more interesting message.

Suppose a factor connects \(X\) and \(Y\):

\[
f(X,Y).
\]

To send a message from the factor to \(X\), the factor:

1. combines itself with the incoming message from \(Y\),
2. eliminates \(Y\),
3. sends the resulting Gaussian over \(X\).

So

\[
m_{f\rightarrow X}(x)
=
\int
f(x,y)
m_{Y\rightarrow f}(y)
\,dy.
\]

This is exactly a local Gaussian Variable Elimination step.



## 6.1 Canonical Pairwise Factor

Represent a pairwise Gaussian factor using

\[
\Lambda_f
=
\begin{bmatrix}
\Lambda_{xx} & \Lambda_{xy}\\
\Lambda_{yx} & \Lambda_{yy}
\end{bmatrix}
\]

and

\[
\eta_f
=
\begin{bmatrix}
\eta_x\\
\eta_y
\end{bmatrix}.
\]

Suppose an incoming message from \(Y\) has canonical parameters

\[
(\eta_m,\Lambda_m).
\]

That message contributes only to the \(Y\) block:

\[
\Lambda_{yy}
\leftarrow
\Lambda_{yy}+\Lambda_m,
\]

\[
\eta_y
\leftarrow
\eta_y+\eta_m.
\]

Then \(Y\) is eliminated using the Schur complement.



## 6.2 Resulting Message

The factor-to-\(X\) message is

\[
\boxed{
\Lambda_{f\rightarrow X}
=
\Lambda_{xx}
-
\Lambda_{xy}
\Lambda_{yy}^{-1}
\Lambda_{yx}
}
\]

and

\[
\boxed{
\eta_{f\rightarrow X}
=
\eta_x
-
\Lambda_{xy}
\Lambda_{yy}^{-1}
\eta_y.
}
\]

These are exactly the same formulas used in canonical Gaussian variable elimination.



## 6.3 Factor-to-Variable Flowchart

```text
Incoming message from Y
          │
          ▼
     Pairwise factor
        f(X,Y)
          │
          ▼
Add information
to Y block
          │
          ▼
Eliminate Y
(Schur complement)
          │
          ▼
Message over X
```


In [ ]:

@dataclass
class PairwiseGaussianFactor:
    variable_1: str
    variable_2: str
    information: np.ndarray
    precision: np.ndarray

    def __post_init__(self) -> None:
        self.information = np.asarray(
            self.information,
            dtype=float,
        )

        self.precision = np.asarray(
            self.precision,
            dtype=float,
        )

        if self.information.shape != (2,):
            raise ValueError(
                "information must have shape (2,)."
            )

        if self.precision.shape != (2, 2):
            raise ValueError(
                "precision must have shape (2, 2)."
            )

        if not np.allclose(
            self.precision,
            self.precision.T,
        ):
            raise ValueError(
                "precision must be symmetric."
            )

    @property
    def variables(self) -> tuple[str, str]:
        return (
            self.variable_1,
            self.variable_2,
        )


In [ ]:

def factor_to_variable_message(
    factor: PairwiseGaussianFactor,
    target_variable: str,
    incoming_message: GaussianBPMessage,
) -> GaussianBPMessage:
    if target_variable not in factor.variables:
        raise ValueError(
            "target_variable is not connected "
            "to the factor."
        )

    if target_variable == factor.variable_1:
        target_index = 0
        eliminate_index = 1
    else:
        target_index = 1
        eliminate_index = 0

    eta = factor.information.copy()
    Lambda = factor.precision.copy()

    eta[eliminate_index] += (
        incoming_message.information
    )

    Lambda[
        eliminate_index,
        eliminate_index,
    ] += incoming_message.precision

    Lambda_aa = Lambda[
        target_index,
        target_index,
    ]

    Lambda_ab = Lambda[
        target_index,
        eliminate_index,
    ]

    Lambda_ba = Lambda[
        eliminate_index,
        target_index,
    ]

    Lambda_bb = Lambda[
        eliminate_index,
        eliminate_index,
    ]

    eta_a = eta[target_index]
    eta_b = eta[eliminate_index]

    if np.isclose(Lambda_bb, 0.0):
        raise ValueError(
            "Cannot eliminate a variable with zero "
            "effective precision."
        )

    outgoing_precision = (
        Lambda_aa
        - Lambda_ab
        * Lambda_ba
        / Lambda_bb
    )

    outgoing_information = (
        eta_a
        - Lambda_ab
        * eta_b
        / Lambda_bb
    )

    return GaussianBPMessage(
        information=outgoing_information,
        precision=outgoing_precision,
    )



# 7. Unary Gaussian Factors

A unary factor represents direct information about one variable.

For example,

\[
X\sim\mathcal{N}(\mu,\sigma^2).
\]

Its canonical parameters are

\[
\Lambda
=
\frac{1}{\sigma^2},
\]

\[
\eta
=
\Lambda\mu.
\]

A unary factor can send this information directly to its variable.


In [ ]:

def unary_gaussian_message(
    mean: float,
    variance: float,
) -> GaussianBPMessage:
    if variance <= 0.0:
        raise ValueError(
            "variance must be positive."
        )

    precision = 1.0 / variance

    information = (
        precision * mean
    )

    return GaussianBPMessage(
        information=information,
        precision=precision,
    )



# 8. Computing a Variable Belief

After a variable receives messages from all neighboring factors, its belief is proportional to their product:

\[
b_X(x)
\propto
\prod_{f\in N(X)}
m_{f\rightarrow X}(x).
\]

In canonical form,

\[
\boxed{
\Lambda_X
=
\sum_{f\in N(X)}
\Lambda_{f\rightarrow X}
}
\]

and

\[
\boxed{
\eta_X
=
\sum_{f\in N(X)}
\eta_{f\rightarrow X}.
}
\]

Then convert to moment form:

\[
\sigma_X^2
=
\frac{1}{\Lambda_X},
\]

\[
\mu_X
=
\frac{\eta_X}{\Lambda_X}.
\]


In [ ]:

def compute_belief(
    incoming_messages: list[GaussianBPMessage],
) -> tuple[float, float]:
    combined = variable_to_factor_message(
        incoming_messages
    )

    return combined.to_moment()



# 9. Worked Example: Gaussian Chain

Consider the tree

```text
prior X        prior Z
   │              │
   ▼              ▼
   X ─── fXY ─── Y ─── fYZ ─── Z
```

We will:

1. send information from \(X\) toward \(Y\),
2. send information from \(Z\) toward \(Y\),
3. combine the incoming messages at \(Y\),
4. compute the marginal belief of \(Y\).

This is the Gaussian analogue of the message-passing example used for discrete Belief Propagation.


In [ ]:

prior_x = unary_gaussian_message(
    mean=0.0,
    variance=1.0,
)

prior_z = unary_gaussian_message(
    mean=4.0,
    variance=1.5,
)

factor_xy = PairwiseGaussianFactor(
    variable_1="X",
    variable_2="Y",
    information=np.array([
        0.0,
        0.0,
    ]),
    precision=np.array([
        [2.0, -1.0],
        [-1.0, 2.0],
    ]),
)

factor_yz = PairwiseGaussianFactor(
    variable_1="Y",
    variable_2="Z",
    information=np.array([
        0.0,
        0.0,
    ]),
    precision=np.array([
        [2.0, -0.8],
        [-0.8, 1.6],
    ]),
)



## 9.1 Message from \(X\) toward \(Y\)

The prior on \(X\) acts as the variable-to-factor message entering \(f_{XY}\).

Then \(f_{XY}\) eliminates \(X\) and sends a Gaussian message to \(Y\).


In [ ]:

message_xy_to_y = factor_to_variable_message(
    factor=factor_xy,
    target_variable="Y",
    incoming_message=prior_x,
)

print(message_xy_to_y)

print(
    "Moment form:",
    message_xy_to_y.to_moment(),
)



## 9.2 Message from \(Z\) toward \(Y\)


In [ ]:

message_yz_to_y = factor_to_variable_message(
    factor=factor_yz,
    target_variable="Y",
    incoming_message=prior_z,
)

print(message_yz_to_y)

print(
    "Moment form:",
    message_yz_to_y.to_moment(),
)



## 9.3 Belief at \(Y\)

Now multiply the two incoming messages.

In canonical form, that means adding their information and precision.


In [ ]:

belief_y_mean, belief_y_variance = (
    compute_belief([
        message_xy_to_y,
        message_yz_to_y,
    ])
)

print("Belief mean at Y:")
print(belief_y_mean)

print("\nBelief variance at Y:")
print(belief_y_variance)



# 10. Message Passing Schedule

On a tree, a convenient schedule is:

```text
Leaves
  │
  ▼
Pass messages inward
  │
  ▼
Choose root
  │
  ▼
Compute root belief
  │
  ▼
Pass messages outward
  │
  ▼
Compute all node beliefs
```

Each directed edge needs one message in each direction.

Once both passes are complete, every variable has enough information to compute its exact marginal.



# 11. Gaussian BP as Local Variable Elimination

This is the most important conceptual connection.

A factor-to-variable message performs:

```text
Multiply local factor
with incoming messages
        │
        ▼
Eliminate every variable
except destination
        │
        ▼
Send reduced factor
as a message
```

That is exactly one local Variable Elimination operation.

So:

> **Gaussian Belief Propagation is Variable Elimination organized as local reusable messages.**



# 12. Minimal Belief Propagation Class

The following class stores the messages needed for a simple scalar pairwise tree.

It is intentionally lightweight because the main purpose of the notebook is to explain the inference process.


In [ ]:

class GaussianBeliefPropagation:
    def __init__(self) -> None:
        self.messages: dict[
            tuple[str, str],
            GaussianBPMessage,
        ] = {}

    def set_message(
        self,
        source: str,
        target: str,
        message: GaussianBPMessage,
    ) -> None:
        self.messages[
            (source, target)
        ] = message

    def get_message(
        self,
        source: str,
        target: str,
    ) -> GaussianBPMessage:
        return self.messages[
            (source, target)
        ]

    def variable_to_factor(
        self,
        incoming_messages: list[
            GaussianBPMessage
        ],
    ) -> GaussianBPMessage:
        return variable_to_factor_message(
            incoming_messages
        )

    def factor_to_variable(
        self,
        factor: PairwiseGaussianFactor,
        target_variable: str,
        incoming_message: GaussianBPMessage,
    ) -> GaussianBPMessage:
        return factor_to_variable_message(
            factor=factor,
            target_variable=target_variable,
            incoming_message=incoming_message,
        )

    def belief(
        self,
        incoming_messages: list[
            GaussianBPMessage
        ],
    ) -> tuple[float, float]:
        return compute_belief(
            incoming_messages
        )



# 13. Verification Against Direct Gaussian Inference

For tree-structured Gaussian models, Belief Propagation must agree with exact inference.

The verification principle is:

```text
Gaussian BP result
        │
        ▼
Compare with
joint Gaussian marginal
or Gaussian Variable Elimination
        │
        ▼
Same mean and variance
```

The reason is not approximation quality.

On a tree, BP is exact because every piece of information is propagated exactly once along each edge without cycles.



# 14. Variable Elimination vs Gaussian Belief Propagation

| Property | Gaussian Variable Elimination | Gaussian Belief Propagation |
|---|---|---|
| Main idea | Eliminate hidden variables sequentially | Exchange local messages |
| Computation style | More centralized | Local / distributed |
| Core operation | Multiply + eliminate | Multiply + eliminate inside messages |
| Canonical form useful? | Yes | Yes |
| Exact on trees? | Yes | Yes |
| Elimination order needed? | Yes | No explicit global order |
| Intermediate large factors | Can appear | Often avoided through local messages |
| Parallelization | Limited | Natural |
| Reusing intermediate information | Less direct | Messages can be reused |
| Loopy graphs | VE still exact if tractable | Loopy BP becomes iterative / approximate |

The two methods are not unrelated algorithms.

They are two organizations of the same underlying Gaussian algebra.



# 15. Final Message-Passing Flowchart

```text
                Gaussian Factor Graph
                         │
                         ▼
                Initialize messages
                         │
                         ▼
              Variable → Factor
         add incoming canonical info
                         │
                         ▼
               Factor → Variable
          combine + eliminate locally
                         │
                         ▼
                  Repeat on tree
                         │
                         ▼
                Incoming messages
                at each variable
                         │
                         ▼
                   Add η and Λ
                         │
                         ▼
                 Convert to μ, Σ
                         │
                         ▼
                 Marginal beliefs
```



# 16. Inference in Gaussian Models: Chapter Summary

We can now see the entire Gaussian inference story as one connected progression.

```text
Multivariate Gaussian
        │
        ▼
Gaussian representation
(μ, Σ)
        │
        ▼
Core Gaussian operations
├── Marginalization
├── Conditioning
└── Product
        │
        ▼
Linear Gaussian models
A x + b + ε
        │
        ▼
Canonical / information form
(η, Λ)
        │
        ▼
Exact Gaussian inference
Variable Elimination
        │
        ▼
Gaussian Belief Propagation
        │
        ▼
Inference in Gaussian Models
COMPLETE
```

The important conceptual lesson is:

> The inference questions remain the same across discrete and Gaussian models.

What changes is the representation and the algebra used to manipulate the factors.



# 17. Where We Go Next

This completes the **Inference** portion of Part I: Probabilistic Reasoning.

The next major topic is **Parameter Learning**.

The question changes from

> Given a model, what should we believe?

to

> Given data, how do we learn the parameters of the probabilistic model?

That will begin the next stage of Part I.
